# fase_1 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 1.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration
Connected to future database: DB_FUTURE


## 2. Ambil Data dari DB Lama

## 3. Helper

In [3]:
from IPython.display import display

def fetch_df(query):
    return pd.read_sql(query, db_old)

def check_nulls(df):
    print("\nNULL CHECK:")
    display(df.isnull().sum())

def check_duplicates(df, subset_cols):
    dup = df[df.duplicated(subset=subset_cols)]
    print(f"\nDUPLICATE ROWS: {len(dup)}")
    display(dup.head())

def preview_df(df, title="Preview", limit=10):
    print(f"\n{title}:")
    display(df.head(limit))

def compare_count(table_old, table_new):
    old = pd.read_sql(f"SELECT COUNT(*) as total FROM {table_old}", db_old)['total'][0]
    new = pd.read_sql(f"SELECT COUNT(*) as total FROM {table_new}", db_new)['total'][0]

    print(f"\nCOUNT CHECK → OLD: {old} | NEW: {new}")
    print("STATUS:", "OK ✅" if old == new else "CHECK ⚠️")

def check_dtype(df, table_name):
    print(f"\n=== CHECK TIPE DATA: {table_name} ===")

    db_schema = pd.read_sql(f"DESCRIBE {table_name}", db_new)

    for col in df.columns:
        df_type = df[col].dtype

        db_type = db_schema[db_schema['Field'] == col]['Type'].values
        db_type = db_type[0] if len(db_type) > 0 else "NOT FOUND"

        print(f"{col} → DF: {df_type} | DB: {db_type}")

In [4]:
def report_not_null_violations(df, table_name):
    print(f"\n=== NOT NULL VIOLATION: {table_name} ===")

    schema = pd.read_sql(f"DESCRIBE {table_name}", db_new)

    violations = {}

    for _, row in schema.iterrows():
        col = row['Field']
        is_nullable = row['Null']

        if col in df.columns and is_nullable == 'NO':
            null_count = df[col].isnull().sum()

            if null_count > 0:
                violations[col] = null_count

    if not violations:
        print("✅ Semua kolom NOT NULL aman")
        return False
    else:
        print("⚠️ Kolom NOT NULL yang bermasalah:")
        for k, v in violations.items():
            print(f"{k}: {v} NULL")
        return True
    
def enforce_not_null(df, table_name):
    schema = pd.read_sql(f"DESCRIBE {table_name}", db_new)

    for _, row in schema.iterrows():
        col = row['Field']
        is_nullable = row['Null']
        col_type = row['Type']

        if col in df.columns and is_nullable == 'NO':
            if df[col].isnull().sum() > 0:

                if 'int' in col_type:
                    df[col] = df[col].fillna(0)
                elif 'date' in col_type or 'time' in col_type:
                    df[col] = df[col].fillna(pd.Timestamp('1970-01-01'))
                else:
                    df[col] = df[col].fillna('Unknown')

    return df


In [5]:
def print_summary(df, table_name):
    """Menampilkan ringkasan dataframe setelah transform"""
    
    print(f"\n{'='*60}")
    print(f"📊 RINGKASAN: {table_name.upper()}")
    print(f"{'='*60}")
    
    print(f"\n📈 DIMENSI DATA:")
    print(f"   • Baris: {len(df):,}")
    print(f"   • Kolom: {len(df.columns)}")
    
    print(f"\n📋 KOLOM: {list(df.columns)}")
    
    print(f"\n🔍 TIPE DATA:")
    for col in df.columns:
        print(f"   • {col}: {df[col].dtype}")
    
    print(f"\n⚠️  NULL COUNT:")
    null_info = df.isnull().sum()
    has_null = False
    for col, count in null_info.items():
        if count > 0:
            print(f"   • {col}: {count} NULL")
            has_null = True
    if not has_null:
        print(f"   ✅ Tidak ada NULL")
    
    print(f"\n📐 STATISTIK:")
    print(f"   • Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
    
    print(f"\n👀 PREVIEW DATA (5 baris pertama):")
    display(df.head())
    print(f"{'='*60}\n")

## 4. Migrating

### Cara Menampilkan Tabel DataFrame

Ada beberapa cara untuk menampilkan dataframe sebagai tabel di Jupyter Notebook:

1. **`display(df)`** - Menampilkan tabel HTML yang rapi ✅ (Recommended)
2. **`df`** - Menampilkan last expression (hanya jika di akhir cell)
3. **`df.head()`** - Menampilkan 5 baris pertama
4. **`df.tail()`** - Menampilkan 5 baris terakhir
5. **`print(df.to_string())`** - Menampilkan sebagai text (kurang bagus)
6. **`df.to_html()`** - Menghasilkan HTML string


In [6]:
# from IPython.display import display

# def migrate_kursus():
#     print("\n=== MIGRATING KURSUS ===")

#     # 1. EXTRACT
#     df_old = fetch_df("""
#         SELECT idpendkursus, nama_kursus, keterangan
#         FROM pendidikankursus
#     """)

#     print("\n📥 DATA ASLI")
#     display(df_old.head())

#     # 2. TRANSFORM
#     df = df_old.rename(columns={
#         'idpendkursus': 'id_kursus',
#         'nama_kursus': 'nama_kursus',
#         'keterangan': 'deskripsi'
#     })

#     # 3. CLEANING
#     df['deskripsi'] = df['deskripsi'].fillna('')
#     df['id_kursus'] = df['id_kursus'].astype(str)

#     # 4. VALIDASI
#     check_dtype(df, "kursus")
#     check_nulls(df)
#     check_duplicates(df, ['id_kursus'])

#     # 5. PREVIEW HASIL TRANSFORM
#     print("\n📤 SETELAH TRANSFORM")
#     display(df.head())

#     return df

In [7]:
# Cek struktur tabel kursus di db_future
print("=== STRUKTUR TABEL kursus (db_future) ===")
display(pd.read_sql("DESCRIBE kursus", db_future))

# Cek juga apakah ada foreign key
fk_query = """
SELECT 
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME
FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
WHERE TABLE_NAME = 'kursus' 
AND REFERENCED_TABLE_NAME IS NOT NULL
"""
fk_info = pd.read_sql(fk_query, db_future)
if not fk_info.empty:
    print("\n=== FOREIGN KEY di kursus ===")
    display(fk_info)
else:
    print("\n✅ Tidak ada foreign key di kursus")

=== STRUKTUR TABEL kursus (db_future) ===


,Field,Type,Null,Key,Default,Extra
0,id_kursus,varchar(15),NO,PRI,NaN,
1,nama_kursus,varchar(150),NO,,NaN,
2,deskripsi,text,NO,,NaN,
3,tipe_kursus,"enum('B2C','B2B')",NO,,NaN,
4,status_arsip,tinyint(1),NO,,0,



=== FOREIGN KEY di kursus ===


,COLUMN_NAME,REFERENCED_TABLE_NAME,REFERENCED_COLUMN_NAME
0,idusers,users,idusers
1,idusers,users,idusers
2,idusers,users,idusers
3,idusers,users,idusers


In [8]:
# =========================================================
# TRANSFORMASI TABEL: kursus (sumber: pendidikankursus)
# =========================================================
print("⚡ Melakukan transformasi tabel 'kursus'...")

# 1. Ambil data dari db_old
df_kursus_old = pd.read_sql("""
    SELECT idpendkursus, nama_kursus, keterangan
    FROM pendidikankursus
""", db_old)
print(f"  Data mentah: {len(df_kursus_old)} baris")
display(df_kursus_old.head())

# 2. Filter: hapus idpendkursus = 'K00017'
df_kursus_old = df_kursus_old[df_kursus_old['idpendkursus'] != 'K00017']
print(f"  Setelah filter hapus K00017: {len(df_kursus_old)} baris")

# 3. Buat dataframe baru
df_kursus = pd.DataFrame()
df_kursus['id_kursus'] = df_kursus_old['idpendkursus'].astype(str).str.strip()
df_kursus['nama_kursus'] = df_kursus_old['nama_kursus'].astype(str).str.strip()
df_kursus['deskripsi'] = df_kursus_old['keterangan'].fillna('').astype(str).str.strip()

# 4. Tentukan tipe_kursus berdasarkan AWALAN nama_kursus
def get_tipe_kursus(nama):
    nama = str(nama).strip()
    if nama.startswith('Kemitraan'):
        return 'B2B'
    elif nama.startswith('LEAP'):
        return 'B2C'
    else:
        return 'B2C'  # default

df_kursus['tipe_kursus'] = df_kursus_old['nama_kursus'].apply(get_tipe_kursus)

# 5. Kolom status_arsip default 0
df_kursus['status_arsip'] = 0

# 6. Validasi: cek null dan duplikat
print("  Cek null:")
print(df_kursus.isnull().sum())

if df_kursus['id_kursus'].duplicated().any():
    print("  ⚠️ Ada duplikasi id_kursus, di-drop.")
    df_kursus = df_kursus.drop_duplicates(subset=['id_kursus'])

print(f"✓ Tabel 'kursus' siap. Shape: {df_kursus.shape}")
print("  Kolom:", list(df_kursus.columns))
display(df_kursus.head())

⚡ Melakukan transformasi tabel 'kursus'...


  Data mentah: 21 baris


,idpendkursus,nama_kursus,keterangan
0,K00001,LEAP - General English,"GE, Balloons, Gogo, SO, Winner"
1,K00002,LEAP - Coding Class,Coding Class Regular
2,K00003,LEAP - Leap Literacy Club,LLC
3,K00004,LEAP - Conversation Class,Conversation Class for Adults
4,K00005,LEAP - Aplikasi Perkantoran,"All In, Private,"


  Setelah filter hapus K00017: 20 baris
  Cek null:
id_kursus       0
nama_kursus     0
deskripsi       0
tipe_kursus     0
status_arsip    0
dtype: int64
✓ Tabel 'kursus' siap. Shape: (20, 5)
  Kolom: ['id_kursus', 'nama_kursus', 'deskripsi', 'tipe_kursus', 'status_arsip']


,id_kursus,nama_kursus,deskripsi,tipe_kursus,status_arsip
0,K00001,LEAP - General English,"GE, Balloons, Gogo, SO, Winner",B2C,0
1,K00002,LEAP - Coding Class,Coding Class Regular,B2C,0
2,K00003,LEAP - Leap Literacy Club,LLC,B2C,0
3,K00004,LEAP - Conversation Class,Conversation Class for Adults,B2C,0
4,K00005,LEAP - Aplikasi Perkantoran,"All In, Private,",B2C,0


In [9]:
print("=== SHOW COLUMNS FROM level ===")
display(pd.read_sql("SHOW COLUMNS FROM level", db_future))

=== SHOW COLUMNS FROM level ===


,Field,Type,Null,Key,Default,Extra
0,id_level,varchar(15),NO,PRI,None,
1,nama_level,varchar(100),NO,,None,
2,urutan_level,int(11),NO,,None,


In [10]:
# def migrate_level():
#     print("\n=== MIGRATING level ===")

#     # 1. EXTRACT
#     df_old = fetch_df("""
#         SELECT idlevel, level, tingkatan
#         FROM level
#     """)

#     preview_df(df_old, "DATA ASLI")

#     # 2. TRANSFORM (mapping kolom)
#     df = df_old.rename(columns={
#         'idlevel': 'id_level',
#         'level': 'nama_level',
#         'tingkatan': 'urutan_level'
#     })

#     # 3. CLEANING

#     ## handle null
#     df['nama_level'] = df['nama_level'].fillna('Unknown')

#     # cek pelanggaran NOT NULL
#     violation = report_not_null_violations(df, "level")

#     ## pastikan tipe
#     df['id_level'] = df['id_level'].astype(str)
#     check_dtype(df, "level")

#     # 4. VALIDASI
#     check_nulls(df)
#     check_duplicates(df, ['id_level'])

#     # cek data desimal di urutan_level
#     display(df[df['urutan_level'] % 1 != 0])

#     # fix tipe urutan_level
#     df['urutan_level'] = df['urutan_level'].fillna(0)
#     df['urutan_level'] = df['urutan_level'].astype(int)

#     check_dtype(df, "level")

#     # 5. PREVIEW HASIL TRANSFORM
#     preview_df(df, "SETELAH TRANSFORM")

#     return df

In [11]:
# =========================================================
# TRANSFORMASI TABEL: level (sumber: level)
# =========================================================
print("⚡ Melakukan transformasi tabel 'level'...")

# 1. Ambil data dari db_old
df_level_old = pd.read_sql("""
    SELECT idlevel, level, tingkatan
    FROM level
""", db_old)
print(f"  Data mentah: {len(df_level_old)} baris")
display(df_level_old.head())

# 2. Buat dataframe baru
df_level = pd.DataFrame()

# 3. Mapping kolom
df_level['id_level'] = df_level_old['idlevel'].astype(str).str.strip()
df_level['nama_level'] = df_level_old['level'].astype(str).str.strip()
df_level['urutan_level'] = df_level_old['tingkatan']

# 4. Cleaning
# nama_level: null -> 'Unknown'
df_level['nama_level'] = df_level['nama_level'].fillna('Unknown')

# urutan_level: handle null -> 0, lalu konversi ke int
# Tapi jika ada desimal (misal 1.5), kita bulatkan? Kode lama pakai fillna(0) lalu astype(int) yang akan error jika ada float.
# Sebaiknya kita cek dulu
print("\n  Cek nilai unik urutan_level sebelum cleaning:")
print(df_level['urutan_level'].unique()[:10])

# Cek apakah ada nilai float dengan desimal bukan .0
decimal_mask = df_level['urutan_level'] % 1 != 0
if decimal_mask.any():
    print(f"  ⚠️ Ada {decimal_mask.sum()} baris dengan nilai desimal:")
    display(df_level[decimal_mask])
    # Konversi: misal bulatkan ke bawah atau ke atas? Asumsikan floor
    df_level.loc[decimal_mask, 'urutan_level'] = df_level.loc[decimal_mask, 'urutan_level'].apply(lambda x: int(x) if not pd.isna(x) else 0)

# Isi null dengan 0
df_level['urutan_level'] = df_level['urutan_level'].fillna(0)

# Konversi ke int
df_level['urutan_level'] = df_level['urutan_level'].astype(int)

# 5. Validasi duplikat id_level
if df_level['id_level'].duplicated().any():
    print("  ⚠️ Ada duplikasi id_level, di-drop (pertama dipertahankan).")
    df_level = df_level.drop_duplicates(subset=['id_level'])

# 6. Cek null (seharusnya tidak ada karena sudah diisi)
print("\n  Cek null setelah cleaning:")
print(df_level.isnull().sum())

# 7. Tampilkan info
print(f"\n✓ Tabel 'level' siap. Shape: {df_level.shape}")
print("  Kolom:", list(df_level.columns))
display(df_level.head())

⚡ Melakukan transformasi tabel 'level'...
  Data mentah: 181 baris


,idlevel,level,tingkatan
0,L00001,Balloons 1A,1.0
1,L00002,Balloons 1B,2.0
2,L00003,Balloons 1C,3.0
3,L00004,Balloons 2A,4.0
4,L00005,Balloons 2B,5.0



  Cek nilai unik urutan_level sebelum cleaning:
[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10.]

  Cek null setelah cleaning:
id_level        0
nama_level      0
urutan_level    0
dtype: int64

✓ Tabel 'level' siap. Shape: (181, 3)
  Kolom: ['id_level', 'nama_level', 'urutan_level']


,id_level,nama_level,urutan_level
0,L00001,Balloons 1A,1
1,L00002,Balloons 1B,2
2,L00003,Balloons 1C,3
3,L00004,Balloons 2A,4
4,L00005,Balloons 2B,5


In [12]:
print("=== SHOW COLUMNS FROM Sesi ===")
display(pd.read_sql("SHOW COLUMNS FROM Sesi", db_future))

=== SHOW COLUMNS FROM Sesi ===


,Field,Type,Null,Key,Default,Extra
0,id_sesi,varchar(15),NO,PRI,None,
1,nama_sesi,varchar(100),NO,,None,
2,waktu_mulai,time,NO,,None,
3,waktu_selesai,time,NO,,None,


In [13]:
# from IPython.display import display

# def migrate_sesi():
#     print("\n=== MIGRATING SESI ===")

#     # 1. EXTRACT
#     df_old = fetch_df("""
#         SELECT idsesi, nama_sesi, waktu_awal, waktu_akhir
#         FROM sesi
#     """)

#     preview_df(df_old, "DATA ASLI")

#     # 2. TRANSFORM
#     df = df_old.rename(columns={
#         'idsesi': 'id_sesi',
#         'nama_sesi': 'nama_sesi',
#         'waktu_awal': 'waktu_mulai',
#         'waktu_akhir': 'waktu_selesai'
#     })

#     # 3. CLEANING

#     ## handle null
#     df['nama_sesi'] = df['nama_sesi'].fillna('Tidak Ada')

#     ## strip spasi
#     df['nama_sesi'] = df['nama_sesi'].str.strip()

#     ## tipe id
#     df['id_sesi'] = df['id_sesi'].astype(str)

#     ## handle waktu (biar sesuai TIME / DATETIME)
#     df['waktu_mulai'] = pd.to_timedelta(df['waktu_mulai']).dt.components.apply(
#     lambda x: f"{int(x.hours):02d}:{int(x.minutes):02d}:{int(x.seconds):02d}", axis=1
#     )

#     df['waktu_selesai'] = pd.to_timedelta(df['waktu_selesai']).dt.components.apply(
#     lambda x: f"{int(x.hours):02d}:{int(x.minutes):02d}:{int(x.seconds):02d}", axis=1
#     )

#     # 4. VALIDASI NOT NULL
#     violation = report_not_null_violations(df, "sesi")

#     if violation:
#         print("\n⚠️ APPLY ENFORCE NOT NULL")
#         df = enforce_not_null(df, "sesi")

#     # 5. VALIDASI UMUM
#     check_dtype(df, "sesi")
#     check_nulls(df)
#     check_duplicates(df, ['id_sesi'])

#     # 6. PREVIEW
#     preview_df(df, "SETELAH TRANSFORM")

#     return df

In [17]:
# =========================================================
# TRANSFORMASI TABEL: sesi (sumber: sesi)
# =========================================================
print("⚡ Melakukan transformasi tabel 'sesi'...")

# 1. Ambil data dari db_old dengan kolom yang benar
df_sesi_old = pd.read_sql("""
    SELECT idsesi, nama_sesi, waktu_awal, waktu_akhir
    FROM sesi
""", db_old)
print(f"  Data mentah: {len(df_sesi_old)} baris")
display(df_sesi_old.head())

# 2. Buat dataframe baru
df_sesi = pd.DataFrame()

# 3. Mapping kolom
df_sesi['id_sesi'] = df_sesi_old['idsesi'].astype(str).str.strip()
df_sesi['nama_sesi'] = df_sesi_old['nama_sesi'].astype(str).str.strip()
# Kolom waktu: pastikan tipe time (dari old sudah time, bisa langsung dipakai)
df_sesi['waktu_mulai'] = df_sesi_old['waktu_awal']
df_sesi['waktu_selesai'] = df_sesi_old['waktu_akhir']

# 4. Cleaning: cek null (seharusnya tidak ada karena NOT NULL di old, tapi tetap amankan)
print("\n  Cek null sebelum cleaning:")
print(df_sesi.isnull().sum())

# Jika ada null di nama_sesi, isi 'Unknown'
df_sesi['nama_sesi'] = df_sesi['nama_sesi'].fillna('Unknown')

# Untuk waktu, jika null (misal dari old), kita isi dengan default '00:00:00'
# Tapi karena old NOT NULL, mungkin tidak perlu. Tetap siapkan:
df_sesi['waktu_mulai'] = df_sesi['waktu_mulai'].fillna(pd.Timestamp('00:00:00').time())
df_sesi['waktu_selesai'] = df_sesi['waktu_selesai'].fillna(pd.Timestamp('00:00:00').time())

# 5. Validasi duplikat id_sesi
if df_sesi['id_sesi'].duplicated().any():
    print("  ⚠️ Ada duplikasi id_sesi, di-drop (pertama dipertahankan).")
    df_sesi = df_sesi.drop_duplicates(subset=['id_sesi'])

# 6. Cek tipe data (waktu_mulai dan waktu_selesai harus object/time)
print("\n  Tipe data kolom waktu:")
print(df_sesi[['waktu_mulai', 'waktu_selesai']].dtypes)

# 7. Tampilkan hasil
print(f"\n✓ Tabel 'sesi' siap. Shape: {df_sesi.shape}")
print("  Kolom:", list(df_sesi.columns))
display(df_sesi.head())

⚡ Melakukan transformasi tabel 'sesi'...
  Data mentah: 43 baris


,idsesi,nama_sesi,waktu_awal,waktu_akhir
0,S00001,GE/LLC Sesi 1,0 days 15:45:00,0 days 16:45:00
1,S00002,GE/LLC Sesi 2,0 days 17:00:00,0 days 18:00:00
2,S00003,GE/LLC Sesi 3,0 days 18:15:00,0 days 19:15:00
3,S00004,CC Kids Sesi 1,0 days 10:10:00,0 days 11:10:00
4,S00005,CC Adult Sesi 1,0 days 16:00:00,0 days 17:00:00



  Cek null sebelum cleaning:
id_sesi          0
nama_sesi        0
waktu_mulai      0
waktu_selesai    0
dtype: int64

  Tipe data kolom waktu:
waktu_mulai      timedelta64[us]
waktu_selesai    timedelta64[us]
dtype: object

✓ Tabel 'sesi' siap. Shape: (43, 4)
  Kolom: ['id_sesi', 'nama_sesi', 'waktu_mulai', 'waktu_selesai']


,id_sesi,nama_sesi,waktu_mulai,waktu_selesai
0,S00001,GE/LLC Sesi 1,0 days 15:45:00,0 days 16:45:00
1,S00002,GE/LLC Sesi 2,0 days 17:00:00,0 days 18:00:00
2,S00003,GE/LLC Sesi 3,0 days 18:15:00,0 days 19:15:00
3,S00004,CC Kids Sesi 1,0 days 10:10:00,0 days 11:10:00
4,S00005,CC Adult Sesi 1,0 days 16:00:00,0 days 17:00:00


In [16]:
# Cek semua kolom di tabel 'sesi' dari db_old
print("=== Daftar kolom di tabel sesi (db_old) ===")
df_check = pd.read_sql("SHOW COLUMNS FROM sesi", db_old)
display(df_check)

# Atau ambil 1 baris data untuk lihat isi
print("\n=== Contoh data sesi (db_old) ===")
df_sample = pd.read_sql("SELECT * FROM sesi LIMIT 3", db_old)
display(df_sample)

=== Daftar kolom di tabel sesi (db_old) ===


,Field,Type,Null,Key,Default,Extra
0,idsesi,varchar(6),NO,PRI,None,
1,nama_sesi,varchar(45),NO,,None,
2,waktu_awal,time,NO,,None,
3,waktu_akhir,time,NO,,None,



=== Contoh data sesi (db_old) ===


,idsesi,nama_sesi,waktu_awal,waktu_akhir
0,S00001,GE/LLC Sesi 1,0 days 15:45:00,0 days 16:45:00
1,S00002,GE/LLC Sesi 2,0 days 17:00:00,0 days 18:00:00
2,S00003,GE/LLC Sesi 3,0 days 18:15:00,0 days 19:15:00


In [18]:
print("=== SHOW COLUMNS FROM libur ===")
display(pd.read_sql("SHOW COLUMNS FROM libur", db_future))

=== SHOW COLUMNS FROM libur ===


,Field,Type,Null,Key,Default,Extra
0,id_libur,varchar(20),NO,PRI,None,
1,nama_event,varchar(150),NO,,None,
2,deskripsi_libur,text,NO,,None,
3,tanggal_mulai,date,NO,,None,
4,tanggal_berakhir,date,NO,,None,
5,label_warna,varchar(20),NO,,None,
6,status_libur_program,tinyint(1),NO,,None,


In [ ]:
# def migrate_libur():
#     print("\n=== MIGRATING LIBUR (FIXED - DENGAN DESCRIPTION & COLOR) ===")

#     # 1. EXTRACT (ambil juga description dan color dari db_old)
#     df_old = fetch_df("""
#         SELECT idlibur, title, start, end, description, color
#         FROM libur
#     """)

#     preview_df(df_old, "DATA ASLI DARI DB_OLD")

#     # 2. TRANSFORM (mapping kolom ke db_new)
#     df = df_old.rename(columns={
#         'idlibur': 'id_libur',
#         'title': 'nama_event',
#         'start': 'tanggal_mulai',
#         'end': 'tanggal_berakhir',
#         'description': 'deskripsi_libur',
#         'color': 'label_warna'
#     })

#     # 3. CLEANING

#     # Handle NULL untuk kolom yang boleh diisi default
#     df['nama_event'] = df['nama_event'].fillna('Tidak Ada')
#     df['deskripsi_libur'] = df['deskripsi_libur'].fillna('')
#     df['label_warna'] = df['label_warna'].fillna('fc-event-default')  # default dari aplikasi

#     # Tambah kolom status_libur_program (konstanta 1)
#     df['status_libur_program'] = 1

#     # Pastikan kolom 'sumber' tidak ada (karena tidak dipakai di db_future)
#     if 'sumber' in df.columns:
#         df = df.drop(columns=['sumber'])

#     # Urutan kolom sesuai db_future (tanpa 'sumber')
#     kolom_urutan = ['id_libur', 'nama_event', 'deskripsi_libur', 
#                     'tanggal_mulai', 'tanggal_berakhir', 'label_warna', 
#                     'status_libur_program']
#     df = df[kolom_urutan]

#     # 4. VALIDASI

#     # Cek pelanggaran NOT NULL
#     violation = report_not_null_violations(df, "libur")
#     if violation:
#         print("\n⚠️ APPLY ENFORCE NOT NULL")
#         df = enforce_not_null(df, "libur")

#     # Tipe data: id_libur string, tanggal diubah ke date (biar MySQL terima)
#     df['id_libur'] = df['id_libur'].astype(str).str.strip()
#     df['tanggal_mulai'] = pd.to_datetime(df['tanggal_mulai']).dt.date
#     df['tanggal_berakhir'] = pd.to_datetime(df['tanggal_berakhir']).dt.date

#     # Cek null & duplikat
#     check_nulls(df)
#     check_duplicates(df, ['id_libur'])

#     # Optional: urutkan berdasarkan id_libur
#     df = df.sort_values('id_libur').reset_index(drop=True)

#     # 5. PREVIEW
#     preview_df(df, "SETELAH TRANSFORM (LENGKAP)")

#     return df

In [20]:
# =========================================================
# TRANSFORMASI TABEL: libur (sumber: libur)
# =========================================================
print("⚡ Melakukan transformasi tabel 'libur'...")

# 1. Ambil data dari db_old
df_libur_old = pd.read_sql("""
    SELECT idlibur, title, start, end, description, color
    FROM libur
""", db_old)
print(f"  Data mentah: {len(df_libur_old)} baris")

# 2. Filter: hapus idlibur = 'L00070' (data trial)
df_libur_old = df_libur_old[df_libur_old['idlibur'] != 'L00070']
print(f"  Setelah hapus L00070: {len(df_libur_old)} baris")

display(df_libur_old.head())

# 3. Buat dataframe baru
df_libur = pd.DataFrame()

# 4. Mapping kolom
df_libur['id_libur'] = df_libur_old['idlibur'].astype(str).str.strip()
df_libur['nama_event'] = df_libur_old['title'].astype(str).str.strip()
df_libur['deskripsi_libur'] = df_libur_old['description'].fillna('').astype(str).str.strip()
df_libur['tanggal_mulai'] = pd.to_datetime(df_libur_old['start'], errors='coerce').dt.date
df_libur['tanggal_berakhir'] = pd.to_datetime(df_libur_old['end'], errors='coerce').dt.date
df_libur['label_warna'] = df_libur_old['color'].fillna('fc-event-default').astype(str).str.strip()
df_libur['status_libur_program'] = 1

# 5. Cleaning nama_event null -> 'Tidak Ada'
df_libur['nama_event'] = df_libur['nama_event'].fillna('Tidak Ada')

# 6. Tanggal NaT -> default (misal '2024-01-01')
if df_libur['tanggal_mulai'].isnull().any():
    print("  ⚠️ Ada tanggal_mulai NaT, diisi default '2024-01-01'")
    df_libur['tanggal_mulai'] = df_libur['tanggal_mulai'].fillna(pd.Timestamp('2024-01-01').date())
if df_libur['tanggal_berakhir'].isnull().any():
    print("  ⚠️ Ada tanggal_berakhir NaT, diisi default '2024-01-01'")
    df_libur['tanggal_berakhir'] = df_libur['tanggal_berakhir'].fillna(pd.Timestamp('2024-01-01').date())

# 7. Duplikat id_libur
if df_libur['id_libur'].duplicated().any():
    print("  ⚠️ Duplikasi id_libur, drop duplicates.")
    df_libur = df_libur.drop_duplicates(subset=['id_libur'])

print(f"\n✓ Tabel 'libur' siap. Shape: {df_libur.shape}")
print("  Kolom:", list(df_libur.columns))
display(df_libur.head())

⚡ Melakukan transformasi tabel 'libur'...
  Data mentah: 79 baris
  Setelah hapus L00070: 78 baris


,idlibur,title,start,end,description,color
0,L00005,Libur Nasional,2023-07-19,2023-07-20,Libur Tahun Baru Islam,fc-event-default
1,L00006,Libur Nasional,2023-06-29,2023-06-30,Libur Idul Adha,fc-event-default
2,L00007,Tahun Baru Islam 2023,2023-07-19,2023-07-20,Libur Tahun Baru Islam 2023,fc-event-default
3,L00008,Natal,2023-12-22,2023-12-30,Libur Natal,fc-event-default
4,L00009,Tahun Baru,2024-01-01,2024-01-02,Libur Tahun Baru,fc-event-default



✓ Tabel 'libur' siap. Shape: (78, 7)
  Kolom: ['id_libur', 'nama_event', 'deskripsi_libur', 'tanggal_mulai', 'tanggal_berakhir', 'label_warna', 'status_libur_program']


,id_libur,nama_event,deskripsi_libur,tanggal_mulai,tanggal_berakhir,label_warna,status_libur_program
0,L00005,Libur Nasional,Libur Tahun Baru Islam,2023-07-19,2023-07-20,fc-event-default,1
1,L00006,Libur Nasional,Libur Idul Adha,2023-06-29,2023-06-30,fc-event-default,1
2,L00007,Tahun Baru Islam 2023,Libur Tahun Baru Islam 2023,2023-07-19,2023-07-20,fc-event-default,1
3,L00008,Natal,Libur Natal,2023-12-22,2023-12-30,fc-event-default,1
4,L00009,Tahun Baru,Libur Tahun Baru,2024-01-01,2024-01-02,fc-event-default,1


In [21]:
print("=== SHOW COLUMNS FROM Topik_diskusi ===")
display(pd.read_sql("SHOW COLUMNS FROM Topik_diskusi", db_future))

=== SHOW COLUMNS FROM Topik_diskusi ===


,Field,Type,Null,Key,Default,Extra
0,id_topik_diskusi,bigint(20) unsigned,NO,PRI,None,auto_increment
1,topik_diskusi,varchar(150),NO,,None,
2,deskripsi_topik_diskusi,text,NO,,None,


In [ ]:
# def migrate_topik_diskusi():
#     print("\n=== MIGRATING TOPIK DISKUSI (DENGAN KETERANGAN) ===")

#     # 1. EXTRACT (ambil idtagmd, tag, dan keterangan dari db_old)
#     df_old = fetch_df("""
#         SELECT idtagmd, tag, keterangan
#         FROM tag_materi_diskusi
#     """)

#     preview_df(df_old, "DATA ASLI DARI DB_OLD")

#     # 2. TRANSFORM
#     df = df_old.rename(columns={
#         'idtagmd': 'id_topik_diskusi',
#         'tag': 'topik_diskusi',
#         'keterangan': 'deskripsi_topik_diskusi'   # mapping keterangan → deskripsi_topik_diskusi
#     })

#     # ID akan auto increment → jangan dipakai saat insert
#     df['id_topik_diskusi'] = None

#     # 3. CLEANING

#     # Handle null untuk topik_diskusi
#     df['topik_diskusi'] = df['topik_diskusi'].fillna('Tidak Ada')
#     df['topik_diskusi'] = df['topik_diskusi'].str.strip()

#     # Handle null untuk deskripsi_topik_diskusi: jika null, isi dengan '' (string kosong)
#     # Jika ada isi, tetap dipertahankan
#     df['deskripsi_topik_diskusi'] = df['deskripsi_topik_diskusi'].fillna('')
#     df['deskripsi_topik_diskusi'] = df['deskripsi_topik_diskusi'].astype(str).str.strip()

#     # 4. REMOVE DUPLICATE berdasarkan topik_diskusi (sesuai kode lama)
#     df = df.drop_duplicates(subset=['topik_diskusi'])

#     # 5. VALIDASI (abaikan id karena auto increment)
#     non_id_cols = ['topik_diskusi', 'deskripsi_topik_diskusi']
#     violation = report_not_null_violations(df[non_id_cols], "topik_diskusi")
    
#     if violation:
#         print("\n⚠️ APPLY ENFORCE NOT NULL (NON-ID)")
#         df_fixed = enforce_not_null(df[non_id_cols], "topik_diskusi")
#         df['topik_diskusi'] = df_fixed['topik_diskusi']
#         df['deskripsi_topik_diskusi'] = df_fixed['deskripsi_topik_diskusi']

#     # Cek tipe data (abaikan id)
#     check_dtype(df[non_id_cols], "topik_diskusi")

#     # Cek null & duplikat
#     check_nulls(df[non_id_cols])
#     check_duplicates(df, ['topik_diskusi'])

#     # 6. PREVIEW HASIL FINAL
#     preview_df(df, "SETELAH TRANSFORM (DENGAN KETERANGAN)")

#     return df

In [23]:
# =========================================================
# TRANSFORMASI TABEL: topik_diskusi (sumber: tag_materi_diskusi)
# =========================================================
print("⚡ Melakukan transformasi tabel 'topik_diskusi'...")

# 1. Ambil data dari db_old
df_topik_old = pd.read_sql("""
    SELECT idtagmd, tag, keterangan
    FROM tag_materi_diskusi
""", db_old)
print(f"  Data mentah: {len(df_topik_old)} baris")
display(df_topik_old.head())

# 2. Buat dataframe baru (id_topik_diskusi auto increment, tidak disertakan)
df_topik = pd.DataFrame()
df_topik['topik_diskusi'] = df_topik_old['tag'].fillna('Tidak Ada').astype(str).str.strip()

# 3. Deskripsi: jika null atau kosong -> 'Tidak ada deskripsi'
df_topik['deskripsi_topik_diskusi'] = df_topik_old['keterangan'].fillna('').astype(str).str.strip()
df_topik['deskripsi_topik_diskusi'] = df_topik['deskripsi_topik_diskusi'].replace('', 'Tidak ada deskripsi')

# 4. Hapus duplikat berdasarkan topik_diskusi (karena harus unik)
before = len(df_topik)
df_topik = df_topik.drop_duplicates(subset=['topik_diskusi'])
after = len(df_topik)
if before != after:
    print(f"  ⚠️ Ditemukan {before - after} baris duplikat topik_diskusi, dihapus.")

# 5. Validasi null (seharusnya tidak ada)
print("\n  Cek null:")
print(df_topik.isnull().sum())

# 6. Tampilkan hasil
print(f"\n✓ Tabel 'topik_diskusi' siap. Shape: {df_topik.shape}")
print("  Kolom:", list(df_topik.columns))
display(df_topik.head())

⚡ Melakukan transformasi tabel 'topik_diskusi'...
  Data mentah: 11 baris


,idtagmd,tag,keterangan
0,T00003,Kendala Siswa,"Siswa yang sering membuat onar dikelas, siswa ..."
1,T00004,Kendala Kelas,Situasi dan kondisi kelas yang mengganggu ata...
2,T00005,Kendala Jadwal,Jika terinfo bahwa siswa memiliki jadwal yang...
3,T00006,Ujian Susulan & Remidi,
4,T00007,"Kendala Zoom, Class In, Koneksi & Device",



  Cek null:
topik_diskusi              0
deskripsi_topik_diskusi    0
dtype: int64

✓ Tabel 'topik_diskusi' siap. Shape: (11, 2)
  Kolom: ['topik_diskusi', 'deskripsi_topik_diskusi']


,topik_diskusi,deskripsi_topik_diskusi
0,Kendala Siswa,"Siswa yang sering membuat onar dikelas, siswa ..."
1,Kendala Kelas,Situasi dan kondisi kelas yang mengganggu atau...
2,Kendala Jadwal,Jika terinfo bahwa siswa memiliki jadwal yang ...
3,Ujian Susulan & Remidi,Tidak ada deskripsi
4,"Kendala Zoom, Class In, Koneksi & Device",Tidak ada deskripsi


In [24]:
print("=== SHOW COLUMNS FROM Kursus_Level ===")
display(pd.read_sql("SHOW COLUMNS FROM Kursus_Level", db_future))

=== SHOW COLUMNS FROM Kursus_Level ===


,Field,Type,Null,Key,Default,Extra
0,id_kursus_level,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_kursus,varchar(20),NO,MUL,None,
2,id_level,varchar(20),NO,MUL,None,


In [ ]:
# def prepare_kursus_level():
#     print("\n=== PREPARE KURSUS LEVEL ===")

#     df_old = fetch_df("""
#         SELECT idpendkursus, idlevel
#         FROM level
#     """)

#     df = df_old.rename(columns={
#         'idpendkursus': 'id_kursus',
#         'idlevel': 'id_level'
#     })

#     # cleaning
#     df['id_kursus'] = df['id_kursus'].astype(str)
#     df['id_level'] = df['id_level'].astype(str)  # karena kamu simpan string

#     df = df.dropna(subset=['id_kursus', 'id_level'])
#     df = df.drop_duplicates(subset=['id_kursus', 'id_level'])

#     preview_df(df, "KURSUS LEVEL SIAP INSERT")

#     return df

In [25]:
# =========================================================
# TRANSFORMASI TABEL: kursus_level (sumber: level dari db_old)
# =========================================================
print("⚡ Melakukan transformasi tabel 'kursus_level'...")

# 1. Ambil data dari tabel 'level' di db_old (karena di old, relasi kursus-level ada di sini)
df_kl_old = pd.read_sql("""
    SELECT idpendkursus, idlevel
    FROM level
""", db_old)
print(f"  Data mentah: {len(df_kl_old)} baris")
display(df_kl_old.head())

# 2. Buat dataframe baru
df_kursus_level = pd.DataFrame()
df_kursus_level['id_kursus'] = df_kl_old['idpendkursus'].astype(str).str.strip()
df_kursus_level['id_level'] = df_kl_old['idlevel'].astype(str).str.strip()

# 3. Hapus baris dengan null di salah satu kolom
before = len(df_kursus_level)
df_kursus_level = df_kursus_level.dropna(subset=['id_kursus', 'id_level'])
after = len(df_kursus_level)
if before != after:
    print(f"  ⚠️ Dihapus {before - after} baris karena null.")

# 4. Hapus duplikat kombinasi id_kursus dan id_level
before = len(df_kursus_level)
df_kursus_level = df_kursus_level.drop_duplicates(subset=['id_kursus', 'id_level'])
after = len(df_kursus_level)
if before != after:
    print(f"  ⚠️ Dihapus {before - after} baris duplikat.")

# 5. Validasi: cek apakah ada id_kursus yang tidak ada di df_kursus (opsional)
kursus_valid = set(df_kursus['id_kursus'])  # df_kursus sudah dibuat sebelumnya
invalid_kursus = set(df_kursus_level['id_kursus']) - kursus_valid
if invalid_kursus:
    print(f"  ⚠️ Ada {len(invalid_kursus)} id_kursus tidak valid (tidak ada di tabel kursus): {list(invalid_kursus)[:5]}")
    # Hapus baris dengan id_kursus tidak valid
    df_kursus_level = df_kursus_level[df_kursus_level['id_kursus'].isin(kursus_valid)]

# 6. Validasi id_level (opsional)
level_valid = set(df_level['id_level'])  # df_level sudah dibuat
invalid_level = set(df_kursus_level['id_level']) - level_valid
if invalid_level:
    print(f"  ⚠️ Ada {len(invalid_level)} id_level tidak valid: {list(invalid_level)[:5]}")
    df_kursus_level = df_kursus_level[df_kursus_level['id_level'].isin(level_valid)]

print(f"  Setelah validasi FK: {len(df_kursus_level)} baris")

# 7. Tampilkan hasil (kolom id_kursus_level auto increment, tidak perlu disertakan)
print(f"\n✓ Tabel 'kursus_level' siap. Shape: {df_kursus_level.shape}")
print("  Kolom:", list(df_kursus_level.columns))
display(df_kursus_level.head())

⚡ Melakukan transformasi tabel 'kursus_level'...
  Data mentah: 181 baris


,idpendkursus,idlevel
0,K00001,L00001
1,K00001,L00002
2,K00001,L00003
3,K00001,L00004
4,K00001,L00005


  ⚠️ Ada 1 id_kursus tidak valid (tidak ada di tabel kursus): ['K00017']
  Setelah validasi FK: 177 baris

✓ Tabel 'kursus_level' siap. Shape: (177, 2)
  Kolom: ['id_kursus', 'id_level']


,id_kursus,id_level
0,K00001,L00001
1,K00001,L00002
2,K00001,L00003
3,K00001,L00004
4,K00001,L00005


In [26]:
print("=== SHOW COLUMNS FROM Kursus_libur ===")
display(pd.read_sql("SHOW COLUMNS FROM Kursus_libur", db_future))

=== SHOW COLUMNS FROM Kursus_libur ===


,Field,Type,Null,Key,Default,Extra
0,id_kursus_libur,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_kursus,varchar(20),NO,MUL,None,
2,id_libur,varchar(20),NO,MUL,None,


In [ ]:
# def prepare_kursus_libur():
#     print("\n=== PREPARE KURSUS LIBUR ===")

#     df_old = fetch_df("""
#         SELECT idpendkursus, idlibur
#         FROM libur_pendkursus
#     """)

#     df = df_old.rename(columns={
#         'idpendkursus': 'id_kursus',
#         'idlibur': 'id_libur'
#     })

#     df['id_kursus'] = df['id_kursus'].astype(str)
#     df['id_libur'] = df['id_libur'].astype(str)

#     df = df.dropna(subset=['id_kursus', 'id_libur'])
#     df = df.drop_duplicates(subset=['id_kursus', 'id_libur'])

#     preview_df(df, "KURSUS LIBUR SIAP INSERT")
#     return df

In [27]:
# =========================================================
# TRANSFORMASI TABEL: kursus_libur (sumber: libur_pendkursus)
# =========================================================
print("⚡ Melakukan transformasi tabel 'kursus_libur'...")

# 1. Ambil data dari db_old
df_klibur_old = pd.read_sql("""
    SELECT idpendkursus, idlibur
    FROM libur_pendkursus
""", db_old)
print(f"  Data mentah: {len(df_klibur_old)} baris")
display(df_klibur_old.head())

# 2. Buat dataframe baru
df_kursus_libur = pd.DataFrame()
df_kursus_libur['id_kursus'] = df_klibur_old['idpendkursus'].astype(str).str.strip()
df_kursus_libur['id_libur'] = df_klibur_old['idlibur'].astype(str).str.strip()

# 3. Hapus null
before = len(df_kursus_libur)
df_kursus_libur = df_kursus_libur.dropna(subset=['id_kursus', 'id_libur'])
after = len(df_kursus_libur)
if before != after:
    print(f"  ⚠️ Dihapus {before - after} baris karena null.")

# 4. Hapus duplikat kombinasi id_kursus dan id_libur
before = len(df_kursus_libur)
df_kursus_libur = df_kursus_libur.drop_duplicates(subset=['id_kursus', 'id_libur'])
after = len(df_kursus_libur)
if before != after:
    print(f"  ⚠️ Dihapus {before - after} baris duplikat.")

# 5. Validasi FK ke tabel kursus (df_kursus)
kursus_valid = set(df_kursus['id_kursus'])
invalid_kursus = set(df_kursus_libur['id_kursus']) - kursus_valid
if invalid_kursus:
    print(f"  ⚠️ Ada {len(invalid_kursus)} id_kursus tidak valid (tidak ada di kursus): {list(invalid_kursus)[:5]}")
    df_kursus_libur = df_kursus_libur[df_kursus_libur['id_kursus'].isin(kursus_valid)]

# 6. Validasi FK ke tabel libur (df_libur)
libur_valid = set(df_libur['id_libur'])
invalid_libur = set(df_kursus_libur['id_libur']) - libur_valid
if invalid_libur:
    print(f"  ⚠️ Ada {len(invalid_libur)} id_libur tidak valid (tidak ada di libur): {list(invalid_libur)[:5]}")
    df_kursus_libur = df_kursus_libur[df_kursus_libur['id_libur'].isin(libur_valid)]

print(f"  Setelah validasi FK: {len(df_kursus_libur)} baris")

# 7. Tampilkan hasil (id_kursus_libur auto increment, tidak disertakan)
print(f"\n✓ Tabel 'kursus_libur' siap. Shape: {df_kursus_libur.shape}")
print("  Kolom:", list(df_kursus_libur.columns))
display(df_kursus_libur.head())

⚡ Melakukan transformasi tabel 'kursus_libur'...
  Data mentah: 2 baris


,idpendkursus,idlibur
0,K00001,L00070
1,K00001,L00068


  ⚠️ Ada 1 id_libur tidak valid (tidak ada di libur): ['L00070']
  Setelah validasi FK: 1 baris

✓ Tabel 'kursus_libur' siap. Shape: (1, 2)
  Kolom: ['id_kursus', 'id_libur']


,id_kursus,id_libur
1,K00001,L00068


In [28]:
# =========================================================
# SIMPAN SEMUA DATAFRAME FASE 1 KE PICKLE
# =========================================================
import pickle

# Kumpulkan semua dataframe fase 1
fase_1_data = {
    'kursus': df_kursus,
    'level': df_level,
    'sesi': df_sesi,
    'libur': df_libur,
    'topik_diskusi': df_topik,
    'kursus_level': df_kursus_level,
    'kursus_libur': df_kursus_libur
}

# Simpan ke file .pkl
with open('fase_1_afrida.pkl', 'wb') as f:
    pickle.dump(fase_1_data, f)

print("✅ Dataframe fase 1 disimpan ke fase_1_afrida.pkl")
print("Isi keys:", list(fase_1_data.keys()))

✅ Dataframe fase 1 disimpan ke fase_1_afrida.pkl
Isi keys: ['kursus', 'level', 'sesi', 'libur', 'topik_diskusi', 'kursus_level', 'kursus_libur']


In [29]:
# # RINGKASAN SEMUA TRANSFORMASI
# print("\n" + "="*60)
# print("🎯 SUMMARY TRANSFORMASI DATA FASE 1")
# print("="*60)

# # 1. Kursus
# print("\n[1/5] Migrasi KURSUS")
# df_kursus = migrate_kursus()
# print_summary(df_kursus, "kursus")

# # 2. Level
# print("\n[2/5] Migrasi LEVEL")
# df_level = migrate_level()
# print_summary(df_level, "level")

# # 3. Sesi
# print("\n[3/5] Migrasi SESI")
# df_sesi= migrate_sesi()
# print_summary(df_sesi, "sesi")

# # 4. Libur
# print("\n[4/5] Migrasi LIBUR")
# df_libur = migrate_libur()
# print_summary(df_libur, "libur")

# # 5. Topik Diskusi
# print("\n[5/5] Migrasi TOPIK DISKUSI")
# df_topik = migrate_topik_diskusi()
# print_summary(df_topik, "topik_diskusi")

# # . Kursus Level
# print("\n[6/5] Migrasi KURSUS LEVEL")
# df_kursus_level = prepare_kursus_level()
# print_summary(df_kursus_level, "kursus_level")

# # . Kursus Libur
# print("\n[6/5] Migrasi KURSUS LIBUR")
# df_kursus_libur = prepare_kursus_libur()
# print_summary(df_kursus_libur, "kursus_libur")

# print("\n✅ SEMUA TRANSFORMASI SELESAI")

In [30]:
with open("fase_1_afrida.pkl", "rb") as f:
    data_loaded = pickle.load(f)

print("📦 Isi file:")
for key in data_loaded.keys():
    print(f" - {key}: {data_loaded[key].shape}")

📦 Isi file:
 - kursus: (20, 5)
 - level: (181, 3)
 - sesi: (43, 4)
 - libur: (78, 7)
 - topik_diskusi: (11, 2)
 - kursus_level: (177, 2)
 - kursus_libur: (1, 2)


In [32]:
from IPython.display import display

for key, df in data_loaded.items():
    print(f"\n📊 {key}")
    display(df.head())


📊 kursus


,id_kursus,nama_kursus,deskripsi,tipe_kursus,status_arsip
0,K00001,LEAP - General English,"GE, Balloons, Gogo, SO, Winner",B2C,0
1,K00002,LEAP - Coding Class,Coding Class Regular,B2C,0
2,K00003,LEAP - Leap Literacy Club,LLC,B2C,0
3,K00004,LEAP - Conversation Class,Conversation Class for Adults,B2C,0
4,K00005,LEAP - Aplikasi Perkantoran,"All In, Private,",B2C,0



📊 level


,id_level,nama_level,urutan_level
0,L00001,Balloons 1A,1
1,L00002,Balloons 1B,2
2,L00003,Balloons 1C,3
3,L00004,Balloons 2A,4
4,L00005,Balloons 2B,5



📊 sesi


...

,topik_diskusi,deskripsi_topik_diskusi
0,Kendala Siswa,"Siswa yang sering membuat onar dikelas, siswa ..."
1,Kendala Kelas,Situasi dan kondisi kelas yang mengganggu atau...
2,Kendala Jadwal,Jika terinfo bahwa siswa memiliki jadwal yang ...
3,Ujian Susulan & Remidi,Tidak ada deskripsi
4,"Kendala Zoom, Class In, Koneksi & Device",Tidak ada deskripsi



📊 kursus_level


,id_kursus,id_level
0,K00001,L00001
1,K00001,L00002
2,K00001,L00003
3,K00001,L00004
4,K00001,L00005



📊 kursus_libur


,id_kursus,id_libur
1,K00001,L00068


## 5. Verifikasi Data

## 6. Return Hasil Migrasi untuk migrate_db.py

## 7. Close Connection